# PPO notebook

#### Imports

In [128]:
from __future__ import annotations

import argparse
import json
import logging
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any
import time
import gc
from copy import deepcopy
import zlib

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

from tribes_env import TribesEnv

# Reduce Reuse Recycle
from dqn import JsonStateEncoder

#### Config

In [129]:
@dataclass
class PPOConfig:
    # Environment
    level_file: str = "tribes/levels/SampleLevel.csv"
    game_mode: str = "SCORE"
    seed: int = 123 # randomized later anyway
    compile_first: bool = True

    # State encoder
    encoder_mode: str = "combined"
    engineered_dim: int = 1024
    raw_dim: int = 256
    
    # Action encoder
    action_feature_dim: int = 256

    # Network
    hidden_dim: int = 1024
    residual_blocks: int = 1

    # PPO
    learning_rate: float = 1e-4
    gamma: float = 0.99
    gae_lambda: float = 0.95

    clip_range: float = 0.2
    gradient_clip_norm: float = 1.0

    entropy_coef: float = 0.01
    value_coef: float = 0.5

    ppo_epochs: int = 4
    minibatch_size: int = 256

    # Rollout collection
    rollout_steps: int = 1000 #
    reward_scale: float = 100.0
    
    # Episode limits
    max_steps_per_episode: int = 1000 #

    # Training schedule
    total_iterations: int = 1000

    # Evaluation
    eval_every_iterations: int = 5
    eval_episodes: int = 10
    eval_seed_offset: int = 10000
    eval_max_steps_per_episode: int = 2000 #


    # Checkpoints
    checkpoint_path: str = "checkpoints/ppo.pt"
    resume: bool = False

    # Device
    device: str = "mps" # cpu

#### Env Helper

In [130]:
def encode_legal_actions(
    info,
    action_encoder,
):
    return np.stack([
        action_encoder.encode(a)
        for a in info["legal_actions"]
    ]).astype(np.float32)

class ActionTextEncoder:
    ACTION_TYPES = (
        "RESEARCH_TECH",
        "END_TURN",
        "MOVE",
        "ATTACK",
        "CAPTURE",
        "SPAWN",
        "BUILD",
        "LEVEL_UP",
        "RESOURCE_GATHERING",
        "RECOVER",
        "MAKE_VETERAN",
        "BUILD_ROAD",
        "CLEAR_FOREST",
        "BURN_FOREST",
        "GROW_FOREST",
        "DESTROY",
        "DISBAND",
        "UPGRADE",
        "CONVERT",
        "HEAL_OTHERS",
        "EXAMINE",
    )

    def __init__(self, output_dim=192):
        self.output_dim = output_dim
        self.type_to_index = {
            name: i
            for i, name in enumerate(self.ACTION_TYPES)
        }

    def encode(self, action):
        vector = np.zeros(
            self.output_dim,
            dtype=np.float32,
        )

        action_type = str(
            action.get("type", "")
        )

        text = str(
            action.get("text", "")
        ).lower()

        if (
            action_type in self.type_to_index
            and self.type_to_index[action_type]
            < self.output_dim
        ):
            vector[
                self.type_to_index[action_type]
            ] = 1.0

        offset = min(
            len(self.ACTION_TYPES),
            self.output_dim,
        )

        buckets = max(
            1,
            self.output_dim - offset,
        )

        for token in (text.replace(":", " ").replace(",", " ").split()):
            idx = (offset + zlib.crc32(token.encode("utf-8")) % buckets)
            vector[idx] += 1.0
        
        for start in range(max(0, len(text) - 2)):
            ngram = text[start:start + 3]
            idx = (offset+ zlib.crc32(ngram.encode("utf-8")) % buckets)
            vector[idx] += 0.25

        norm = np.linalg.norm(vector)

        if norm > 0:
            vector /= norm

        return vector

In [131]:
def make_env(config: PPOConfig) -> TribesEnv:
    return TribesEnv(
        level_file=config.level_file,
        game_mode=config.game_mode,
        seed=config.seed,
        compile_first=False,
    )

def evaluate_agent(
    agent,
    config,
    encoder,
    action_encoder,
    episodes,
    seed_offset=0,
):
    env = make_env(config)

    rewards = []

    t0 = time.time()

    print(f"[eval] start episodes={episodes}")

    for episode in range(episodes):

        ep_t0 = time.time()

        obs, info = env.reset(
            seed=config.seed + seed_offset + episode
        )

        state = encoder.encode(obs["state_json"])
        episode_reward = 0.0
        steps = 0

        for _ in range(config.eval_max_steps_per_episode):

            legal_action_features = encode_legal_actions(
                info,
                action_encoder,
            )

            action = agent.select_greedy_action(
                state,
                legal_action_features,
            )

            next_obs, reward, terminated, truncated, next_info = env.step(action)

            episode_reward += reward
            steps += 1

            state = encoder.encode(next_obs["state_json"])
            info = next_info

            if terminated or truncated:
                break

        rewards.append(float(episode_reward))

        ep_time = time.time() - ep_t0

        print(
            f"[eval] episode={episode+1}/{episodes} "
            f"reward={episode_reward:.2f} "
            f"steps={steps} "
            f"time={ep_time:.2f}s"
        )

    env.close()

    mean_reward = float(np.mean(rewards))
    total_time = time.time() - t0

    print(
        f"[eval] done mean_reward={mean_reward:.3f} "
        f"total_time={total_time:.2f}s "
        f"avg_ep_time={total_time/episodes:.2f}s"
    )

    return {
        "episode_rewards": rewards,
        "mean_reward": mean_reward,
    }

#### Model definition

In [132]:
class ResidualBlock(nn.Module):
    def __init__(self, dim: int):
        super().__init__()

        self.fc1 = nn.Linear(dim, dim)
        self.norm1 = nn.LayerNorm(dim)

        self.fc2 = nn.Linear(dim, dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        residual = x

        x = self.fc1(x)
        x = self.norm1(x)
        x = F.gelu(x)

        x = self.fc2(x)
        x = self.norm2(x)

        x = x + residual

        return F.gelu(x)

class PPOModel(nn.Module):
    def __init__(
        self,
        state_dim,
        action_feature_dim,
        hidden_dim,
        residual_blocks
    ):
        super().__init__()

        default_layers = [
            nn.Linear(state_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        ]

        for _ in range(residual_blocks):
            default_layers.append(ResidualBlock(hidden_dim))

        self.state_net = nn.Sequential(*default_layers)

        # residual blocks unused
        self.action_net = nn.Sequential(
            nn.Linear(action_feature_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )

        self.policy_head = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, 1)
        )

        self.value_head = nn.Linear(hidden_dim, 1)

    def forward(self,states, action_features):
        state_emb = self.state_net(states)
        action_emb = self.action_net(action_features)

        state_expanded = state_emb.unsqueeze(1).expand(-1, action_emb.shape[1], -1)
        x = torch.cat([state_expanded, action_emb, state_expanded * action_emb], dim=-1)
        logits = self.policy_head(x).squeeze(-1)
        values = self.value_head(state_emb).squeeze(-1)

        return logits, values
    
    def value(self, states):
        state_emb = self.state_net(states)
        return self.value_head(state_emb).squeeze(-1)

#### Rollout utilities

In [ ]:
def masked_categorical(logits, action_masks,):
    logits = logits.masked_fill( ~action_masks, torch.finfo(logits.dtype).min,)
    return torch.distributions.Categorical(logits=logits)

def compute_gae(rewards, dones, values, last_value, gamma, gae_lambda):
    advantages = []

    gae = 0.0

    values = values + [last_value]

    for t in reversed(
        range(len(rewards))
    ):
        delta = (
            rewards[t]
            + gamma
            * values[t + 1]
            * (1.0 - dones[t])
            - values[t]
        )

        gae = (
            delta
            + gamma
            * gae_lambda
            * (1.0 - dones[t])
            * gae
        )

        advantages.insert(0, gae)

    returns = [
        adv + value
        for adv, value
        in zip(
            advantages,
            values[:-1],
        )
    ]

    return advantages, returns

@dataclass
class RolloutBuffer:
    states: list
    actions: list
    rewards: list
    dones: list

    values: list
    log_probs: list

    action_features: list

    final_state: np.ndarray
    final_mask: np.ndarray

#### PPO Agent Definition

In [ ]:
class PPOAgent:
    def __init__(
        self,
        state_dim,
        config,
    ):
        self.config = config

        self.device = torch.device(
            config.device
        )

        self.model = PPOModel(
            state_dim,
            config.action_feature_dim,
            config.hidden_dim,
            config.residual_blocks,
        ).to(self.device)

        self.optimizer = torch.optim.Adam(
            self.model.parameters(),
            lr=config.learning_rate,
        )

    @torch.no_grad()
    def select_action(
        self,
        state,
        action_features,
    ):
        state_tensor = torch.as_tensor(
            state,
            dtype=torch.float32,
            device=self.device,
        ).unsqueeze(0)

        action_tensor = torch.as_tensor(
            action_features,
            dtype=torch.float32,
            device=self.device,
        ).unsqueeze(0)

        mask_tensor = torch.ones(
            (1, action_features.shape[0]),
            dtype=torch.bool,
            device=self.device,
        )

        logits, values = self.model(
            state_tensor,
            action_tensor,
        )

        dist = masked_categorical(
            logits,
            mask_tensor,
        )

        action = dist.sample()

        return (
            int(action.item()),
            float(dist.log_prob(action).item()),
            float(values.item()),
        )
    
    @torch.no_grad()
    def select_greedy_action(
        self,
        state,
        action_features,
    ):
        state_tensor = torch.as_tensor(
            state,
            dtype=torch.float32,
            device=self.device,
        ).unsqueeze(0)

        action_tensor = torch.as_tensor(
            action_features,
            dtype=torch.float32,
            device=self.device,
        ).unsqueeze(0)

        logits, _ = self.model(
            state_tensor,
            action_tensor,
        )

        action = torch.argmax(
            logits,
            dim=1,
        )

        return int(action.item())
    
    @torch.no_grad()
    def predict_value(self, state):
        state_tensor = torch.as_tensor(
            state,
            dtype=torch.float32,
            device=self.device,
        ).unsqueeze(0)

        value = self.model.value(state_tensor)

        return float(value.squeeze(0).item())

    def ppo_update(
        self,
        states,
        actions,
        old_log_probs,
        returns,
        advantages,
        action_masks,
        action_features,
    ):
        logits, values = self.model(
            states, 
            action_features
        )

        dist = masked_categorical(
            logits,
            action_masks,
        )

        new_log_probs = dist.log_prob(
            actions
        )

        entropy = dist.entropy()

        ratio = torch.exp(
            new_log_probs
            - old_log_probs
        )

        surr1 = ratio * advantages

        surr2 = (
            torch.clamp(
                ratio,
                1.0 - self.config.clip_range,
                1.0 + self.config.clip_range,
            )
            * advantages
        )

        policy_loss = (
            -torch.min(
                surr1,
                surr2,
            ).mean()
        )

        value_loss = F.mse_loss(
            values,
            returns,
        )

        entropy_loss = entropy.mean()

        loss = (
            policy_loss
            + self.config.value_coef
            * value_loss
            - self.config.entropy_coef
            * entropy_loss
        )

        self.optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            self.model.parameters(),
            self.config.gradient_clip_norm,
        )

        self.optimizer.step()

        return {
            "loss": float(loss.item()),
            "policy_loss": float(
                policy_loss.item()
            ),
            "value_loss": float(
                value_loss.item()
            ),
            "entropy": float(
                entropy_loss.item()
            ),
        }

#### Rollout Collection

In [136]:
def collect_rollout(
    env,
    agent,
    encoder, 
    action_encoder, 
    config
):
    obs, info = env.reset()

    state = encoder.encode(obs["state_json"])

    rollout = RolloutBuffer(
        states=[],
        actions=[],
        rewards=[],
        dones=[],
        values=[],
        log_probs=[],
        action_features=[],
        final_state = None,
        final_mask = None
    )

    while (len(rollout.states) < config.rollout_steps):
        legal_action_features = encode_legal_actions(info,action_encoder)

        action, log_prob, value = agent.select_action(state, legal_action_features)
        next_obs, reward, terminated, truncated, next_info = env.step(action)

        rollout.states.append(state)
        rollout.actions.append(action)
        rollout.rewards.append(reward / config.reward_scale)
        rollout.dones.append(terminated or truncated)
        rollout.values.append(value)
        rollout.log_probs.append(log_prob)
        
        rollout.action_features.append(encode_legal_actions(info, action_encoder))
        
        state = encoder.encode(next_obs["state_json"])

        info = next_info

        if terminated or truncated:
            obs, info = env.reset()
            state = encoder.encode(obs["state_json"])
    
    rollout.final_state = state
    rollout.final_mask = np.asarray(
        info["action_mask"],
        dtype=np.bool_,
    )

    return rollout

#### Train Loop

In [137]:
# prep env and checkpoint pathing
config = PPOConfig()

Path(config.checkpoint_path).parent.mkdir(parents=True, exist_ok=True)

# state encoder
encoder = JsonStateEncoder(
    mode="combined",
    engineered_dim=config.engineered_dim,
    raw_dim=config.raw_dim,
)

# action encoder
action_encoder = ActionTextEncoder(
    output_dim=config.action_feature_dim
)

env = make_env(config)
obs, info = env.reset()
state_dim = encoder.encode(obs["state_json"]).shape[0]

best_eval_reward = float("-inf")
start_iteration = 0

base = Path(config.checkpoint_path)
last_path = base.parent / f"{base.stem}_last.pt"
best_path = base.parent / f"{base.stem}_best.pt"

if config.resume and last_path.exists():
    print(f"Loading checkpoint: {last_path}")
    checkpoint = torch.load(last_path, map_location=config.device)

    agent = PPOAgent(state_dim, config)

    agent.model.load_state_dict(checkpoint["model_state_dict"])
    agent.optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    start_iteration = checkpoint.get("iteration", 0)
    best_eval_reward = checkpoint.get("best_eval_reward", float("-inf"))

else:
    agent = PPOAgent(state_dim, config)

checkpoint = {}

In [138]:
# actual train loop
for iteration in range(start_iteration, config.total_iterations):
    iter_time = time.time()
    rollout = collect_rollout(
        env,
        agent,
        encoder,
        action_encoder,
        config,
    )

    # predict value if reaching end of rollout
    last_state = rollout.final_state
    last_value = agent.predict_value(last_state)

    advantages, returns = compute_gae(
        rollout.rewards,
        rollout.dones,
        rollout.values,
        last_value=last_value,
        gamma=config.gamma,
        gae_lambda=config.gae_lambda,
    )

    # as_tensor doesn't copy input data, just reuses existing space
    states = torch.as_tensor(
        np.array(rollout.states),
        dtype=torch.float32,
        device=agent.device,
    )

    max_actions = max(
        af.shape[0]
        for af in rollout.action_features
    )

    action_feature_array = np.zeros(
        (
            len(rollout.action_features),
            max_actions,
            config.action_feature_dim,
        ),
        dtype=np.float32,
    )

    action_mask_array = np.zeros(
        (
            len(rollout.action_features),
            max_actions,
        ),
        dtype=np.bool_,
    )

    for i, af in enumerate(rollout.action_features):
        n = af.shape[0]
        action_feature_array[i, :n] = af
        action_mask_array[i, :n] = True

    action_features = torch.as_tensor(
        action_feature_array,
        dtype=torch.float32,
        device=agent.device,
    )

    action_masks = torch.as_tensor(
        action_mask_array,
        dtype=torch.bool,
        device=agent.device,
    )

    
    actions = torch.as_tensor(
        rollout.actions,
        dtype=torch.long,
        device=agent.device,
    )

    old_log_probs = torch.as_tensor(
        rollout.log_probs,
        dtype=torch.float32,
        device=agent.device,
    )

    returns_t = torch.as_tensor(
        returns,
        dtype=torch.float32,
        device=agent.device,
    )

    advantages_t = torch.as_tensor(
        advantages,
        dtype=torch.float32,
        device=agent.device,
    )

    advantages_t = (advantages_t - advantages_t.mean()) / (advantages_t.std() + 1e-8)

    num_samples = states.shape[0]
    for _ in range(config.ppo_epochs):
        permutation = torch.randperm(num_samples, device=agent.device)
        for start in range(0, num_samples, config.minibatch_size):
            end = start + config.minibatch_size
            idx = permutation[start:end]
            stats = agent.update(
                states[idx],
                actions[idx],
                old_log_probs[idx],
                returns_t[idx],
                advantages_t[idx],
                action_masks[idx],
                action_features[idx],
            )

    # separate iter train time from eval
    iter_time = time.time() - iter_time

    # evaluate agent
    eval_reward = None

    if (config.eval_every_iterations > 0
        and (iteration + 1) % config.eval_every_iterations == 0):
        eval_results = evaluate_agent(
            agent,
            config,
            encoder,
            action_encoder,
            config.eval_episodes,
            seed_offset=(config.eval_seed_offset + iteration),
        )

        eval_reward = (eval_results["mean_reward"])

        # latest checkpointing
        checkpoint = {
            "iteration": iteration + 1,
            "model_state_dict": agent.model.state_dict(),
            "optimizer_state_dict": agent.optimizer.state_dict(),
            "config": asdict(config),
            "best_eval_reward": best_eval_reward,
        }

        print("Saving latest checkpoint")
        torch.save(checkpoint, last_path)

        if eval_reward > best_eval_reward:
            print("Saving best checkpoint")
            best_eval_reward = eval_reward
            checkpoint["best_eval_reward"] = best_eval_reward
            torch.save(checkpoint, best_path)

    print(
        f"iter={iteration + 1}",
        f"loss={stats['loss']:+.4f}",
        f"policy={stats['policy_loss']:+.4f}",
        f"value={stats['value_loss']:+.4f}",
        f"entropy={stats['entropy']:+.4f}",
        f"time (iter)={iter_time:.1f}s"
    )

    # drop references to rollout tensors/lists
    del rollout
    del states, actions, old_log_probs, returns_t, advantages_t, action_masks

    # try and garbage collect
    gc.collect() 
    if agent.device.type == "cuda":
        torch.cuda.empty_cache()

iter=1 loss=+0.7886 policy=+0.0456 value=+1.5429 entropy=+2.8434 time (iter)=44.8s
iter=2 loss=+0.8540 policy=+0.0674 value=+1.6273 entropy=+2.7002 time (iter)=34.4s
iter=3 loss=+1.0368 policy=+0.0037 value=+2.1173 entropy=+2.5544 time (iter)=31.0s
iter=4 loss=+2.0142 policy=-0.0339 value=+4.1442 entropy=+2.3975 time (iter)=29.1s
[eval] start episodes=10
[eval] episode=1/10 reward=12156.00 steps=500 time=7.65s
[eval] episode=2/10 reward=12156.00 steps=500 time=5.14s
[eval] episode=3/10 reward=12082.00 steps=492 time=5.56s
[eval] episode=4/10 reward=11820.00 steps=499 time=5.46s
[eval] episode=5/10 reward=12068.00 steps=490 time=5.18s
[eval] episode=6/10 reward=11634.00 steps=477 time=4.78s
[eval] episode=7/10 reward=11911.00 steps=491 time=5.09s
[eval] episode=8/10 reward=11851.00 steps=468 time=4.74s
[eval] episode=9/10 reward=11851.00 steps=468 time=4.74s
[eval] episode=10/10 reward=12092.00 steps=511 time=5.71s
[eval] done mean_reward=11962.100 total_time=54.08s avg_ep_time=5.41s
Sa

KeyboardInterrupt: 